# QubitX-MaxCut: QAOA for Weighted Max-Cut & SWAP Bottleneck Analysis
**Qiskit Fall Fest 2026 — Phase 1 Hackathon**  
**Team:** QubitX | **Track:** Quantum Optimization | **Problem:** PS-O1 — Max-Cut on Weighted Graphs

---

## 📌 Walkthrough Overview
This notebook presents an end-to-end, fully reproducible demonstration of:
1. Exact ground-truth Max-Cut baseline calculation using a brute-force classical solver.
2. Parameterized QAOA circuit construction, simulation, and parameter tuning using SciPy COBYLA.
3. Transpilation bottleneck analysis on provisional hardware topologies (`processor_A.json` and `processor_B.json`).
4. Graph-size capacity sweep across 5, 6, and 7 logical nodes.
5. Automated generation of publication-quality metrics tables and figures.

> **Provisional Processor Disclaimer:** Processor A and Processor B configurations in `processors/` are provisional development examples used for SWAP bottleneck modeling. Official organizer specifications remain pending.

In [ ]:
# 1. Imports and Environment Setup
import sys
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

from src.graph_utils import create_sample_graph, solve_max_cut_brute_force, create_synthetic_graph
from src.qaoa_builder import build_qaoa_circuit, optimize_qaoa, get_most_frequent_partition
from src.processors import load_development_processors, inspect_processor_transpilation_layouts
from src.benchmarking import run_qaoa_benchmark, run_processor_ab_benchmark, run_graph_size_routing_sweep
from src.visualization import generate_all_plots

print("Environment setup complete. All modules imported successfully.")


## 2. Problem Graph & Classical Ground-Truth Baseline

We construct the 4-node, 5-edge weighted development problem graph and compute the exact ground-truth Max-Cut value using exhaustive brute-force evaluation.

In [ ]:
g_dev = create_sample_graph()
exact_val, best_part, all_opts = solve_max_cut_brute_force(g_dev)

print(f"Development Graph Nodes: {g_dev.number_of_nodes()}, Edges: {g_dev.number_of_edges()}")
print(f"Exact Ground-Truth Max-Cut Value: {exact_val}")
print(f"Optimal Partition Assignment: {best_part}")


## 3. QAOA Circuit Construction & Parameter Optimization

We construct a parameterized QAOA circuit ($p=1$), optimize angles $\gamma$ and $\beta$ using COBYLA on `AerSimulator`, and evaluate expected cut value $\langle C \rangle$.

In [ ]:
# Construct unmeasured QAOA circuit for visualization/analysis
qc_logical = build_qaoa_circuit(g_dev, p=1, measure=False)
print("Logical QAOA Circuit Depth:", qc_logical.depth())
print("Logical Gate Counts:", dict(qc_logical.count_ops()))

# Optimize parameters
opt_params, opt_exp, counts = optimize_qaoa(g_dev, p=1, maxiter=30, shots=1000, seed=42)
top_part, top_val, prob = get_most_frequent_partition(counts, g_dev)

print(f"Optimized QAOA Expected Cut <C>: {opt_exp:.3f}")
print(f"Approximation Ratio (<C> / C_max): {opt_exp / exact_val:.4f}")
print(f"Most Frequent Sampled Partition Cut: {top_val} (Probability: {prob:.3f})")


## 4. Hardware Processor Comparison (Processor A vs Processor B)

We benchmark the same QAOA circuit across `processor_A.json` (5-qubit tree) and `processor_B.json` (7-qubit Heavy-Hex sub-unit).

In [ ]:
df_ab = run_processor_ab_benchmark(g_dev, p_list=[1, 2], maxiter=25, shots=1000, seed_simulator=42, seed_transpiler=42)
print("=== Processor A & Processor B Benchmark Comparison ===")
print(df_ab[['processor_id', 'physical_qubits', 'qaoa_depth_p', 'orig_depth', 'transpiled_depth', 'orig_2q_gates', 'transpiled_2q_gates', 'swap_overhead', 'opt_expected_cut', 'approx_ratio_expected']].to_string())


## 5. Transpilation Layout Diagnostics

We inspect the physical qubit mapping selected by Qiskit's layout pass to explain why routing metrics match on 4-qubit workloads.

In [ ]:
diag = inspect_processor_transpilation_layouts(qc_logical, seed_transpiler=42)
for pid, info in diag.items():
    print(f"=== {pid} ({info['processor_name']}) ===")
    print("Physical Edges:", info['coupling_edges'])
    print("Initial Layout:", info['initial_layout'])
    print("Transpiled Depth:", info['transpiled_depth'])
    print("Gate Counts:", info['op_counts'])


## 6. Graph-Size Capacity & Routing Sweep (5, 6, 7 Nodes)

We run a scaling sweep to evaluate physical qubit capacity limits and routing SWAP inflation on expanding graph sizes.

In [ ]:
df_sweep = run_graph_size_routing_sweep(node_counts=[5, 6, 7], p=1, seed_transpiler=42)
print("=== Graph-Size Routing Sweep Results ===")
print(df_sweep[['logical_nodes', 'logical_edges', 'processor_id', 'physical_qubits', 'is_supported', 'transpiled_depth', 'swap_overhead', 'notes']].to_string())


## 7. Generate Figures & Output Verification

We run `generate_all_plots()` to export publication-quality figures under `results/figures/` and `results/plots/`.

In [ ]:
fig_files = generate_all_plots()
print("Generated Plot Figures:", fig_files)
